[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saketkc/pysradb/blob/develop/notebooks/08.PMC_DOI_Identifiers.ipynb)

# Extract Identifiers from PMC/DOI

This notebook shows how to extract SRA identifiers from PubMed Central articles and DOI references.

In [ ]:
# Install pysradb if not already installed
try:
    import pysradb

    print(f"pysradb {pysradb.__version__} is already installed")
except ImportError:
    print("Installing pysradb from GitHub...")
    import sys

    !{sys.executable} -m pip install -q git+https://github.com/saketkc/pysradb
    print("pysradb installed successfully!")

## Extracting Database Identifiers from Literature (PMC/DOI)

This notebook demonstrates how to extract database identifiers (GSE, PRJNA, SRP, SRR, SRX, SRS) from:
- PubMed Central (PMC) articles
- PubMed IDs (PMIDs)
- Digital Object Identifiers (DOIs)



## Setup

First, let's import pysradb and create a connection:

In [ ]:
from pysradb.sraweb import SRAweb
import pandas as pd

db = SRAweb()

## Example 1: Extract Identifiers from PMID

Let's extract all database identifiers from a PubMed article using its PMID:

In [ ]:
# Extract all identifiers from PMID 39528918
pmid = "39528918"
df = db.pmid_to_identifiers(pmid)

print(f"Identifiers found for PMID {pmid}:")
df

## Example 2: Get Only GSE IDs from PMID

Sometimes you only need specific identifier types:

In [ ]:
# Get only GSE identifiers
gse_df = db.pmid_to_gse("39528918")
print("GSE identifiers:")
gse_df

## Example 3: Get Only SRP IDs from PMID

**Important:** Even if the paper only mentions GSE IDs, pysradb will automatically convert them to SRP!

In [ ]:
# Get only SRP identifiers
# This works even if the paper only mentions GSE253406!
srp_df = db.pmid_to_srp("39528918")
print("SRP identifiers (auto-converted from GSE if needed):")
srp_df

## Example 4: Process Multiple PMIDs

You can batch process multiple PMIDs at once:

In [ ]:
# Process multiple PMIDs
pmids = ["39528918", "27373336"]
df_multiple = db.pmid_to_identifiers(pmids)

print(f"Identifiers from {len(pmids)} PMIDs:")
df_multiple

## Example 5: Extract from PMC ID

If you have a PMC ID directly, you can use it:

In [ ]:
# Extract from PMC ID
pmc_df = db.pmc_to_identifiers("PMC10802650")
print("Identifiers from PMC article:")
pmc_df

## Example 6: Extract from DOI

You can also extract identifiers directly from a DOI:

In [ ]:
# Extract from DOI
doi = "10.12688/f1000research.18676.1"
doi_df = db.doi_to_identifiers(doi)

print(f"Identifiers from DOI {doi}:")
doi_df

## Example 7: Get GSE from DOI

In [ ]:
# Get only GSE from DOI
doi_gse_df = db.doi_to_gse("10.12688/f1000research.18676.1")
print("GSE identifiers from DOI:")
doi_gse_df

## Example 8: Complete Workflow - DOI to Data Download

Here's a complete workflow from DOI to downloading the actual data:

In [ ]:
# Step 1: Extract SRP from DOI
doi = "10.12688/f1000research.18676.1"
print(f"Step 1: Extracting identifiers from DOI: {doi}")
doi_results = db.doi_to_srp(doi)
print(doi_results)
print()

In [ ]:
# Step 2: Extract SRP ID from results
if not doi_results.empty and not pd.isna(doi_results.iloc[0]["srp_ids"]):
    srp_id = doi_results.iloc[0]["srp_ids"].split(",")[0]  # Take first SRP if multiple
    print(f"Step 2: Found SRP ID: {srp_id}")
    print()

    # Step 3: Get metadata for this SRP
    print(f"Step 3: Getting metadata for {srp_id}")
    metadata = db.sra_metadata(srp_id)
    print(f"Found {len(metadata)} samples")
    print(metadata.head())

    # Optional: Download the data
    # db.download(df=metadata, out_dir='./downloads')
else:
    print("No SRP IDs found for this DOI")

## Example 9: Batch Processing Literature

Process multiple papers from a literature search:

In [ ]:
# Simulate PMIDs from a literature search
pmids_from_search = ["39528918", "27373336", "30873266"]

# Extract identifiers from all papers
results = db.pmid_to_identifiers(pmids_from_search)

# Filter to papers with SRA data
papers_with_sra = results[~pd.isna(results["srp_ids"])]

print(f"Processed {len(pmids_from_search)} papers")
print(f"Found {len(papers_with_sra)} papers with SRA data")
print()
print("Papers with SRA data:")
papers_with_sra

## Example 10: Extract Identifiers from Custom Text

You can also extract identifiers from any text:

In [ ]:
# Custom text with database identifiers
text = """
This study analyzed RNA-seq data from GSE81903 and SRP075720.
The BioProject accession is PRJNA319707 with representative samples 
SRR3587529, SRX1800089, and SRS1467635.
"""

identifiers = db.extract_identifiers_from_text(text)

print("Identifiers found in text:")
for id_type, ids in identifiers.items():
    if ids:
        print(f"  {id_type.upper()}: {', '.join(ids)}")

## Example 11: ID Conversion Utilities

Convert between different identifier types:

In [ ]:
# Convert PMID to PMC
print("PMID to PMC conversion:")
pmid_pmc = db.pmid_to_pmc("27373336")
print(pmid_pmc)
print()

# Convert DOI to PMID
print("DOI to PMID conversion:")
doi_pmid = db.doi_to_pmid("10.12688/f1000research.18676.1")
print(doi_pmid)